# K4b · MIL model training, second seed (GPU)
Settings: **Internet ON** (to download ConvNeXt Tiny ImageNet weights), accelerator **GPU T4 x2** or **P100**.
Inputs: competition data, outputs of **K1**, **K2** and **K3**.

* Training on the 4,349 studies with LLM soft targets.
* The 58 gold studies are **never** used for training: gold AUC is measured after each epoch, with a bootstrap at the end.
* The final model is the **last epoch** (EMA of weights), not the best on gold, to avoid overfitting these 58 studies.

Outputs: `knee_model.py`, `model_final.pt`, `config.json`, `prior.json`, `gold_preds.csv`.

Second model for the ensemble: same architecture, different random seed (different window sampling, augmentations and head initialisation). Optional input: the **K4 training** output, to measure the ensemble on the gold studies.

In [ ]:
%%writefile /kaggle/working/knee_model.py
# knee_model.py
# MIL model with per-finding attention, shared by K4 (training) and K5 (inference).
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def build_backbone(arch="convnext_tiny", pretrained=True):
    import timm
    return timm.create_model(arch, pretrained=pretrained, num_classes=0, global_pool="avg", in_chans=3)


class PerFindingAttentionHead(nn.Module):
    """K feature vectors of size F -> 12 logits, one attention per class.
    Softmax runs over windows: each class picks its own windows."""

    def __init__(self, f_dim, n_classes=12, hidden=256, drop=0.2, n_pos=64):
        super().__init__()
        self.pos = nn.Embedding(n_pos, f_dim)        # slice position: tells the model which slot it is looking at
        nn.init.normal_(self.pos.weight, std=0.02)
        self.norm = nn.LayerNorm(f_dim)
        self.att = nn.Sequential(nn.Linear(f_dim, hidden), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(hidden, n_classes))
        self.clsW = nn.Parameter(torch.zeros(n_classes, f_dim))
        self.clsb = nn.Parameter(torch.zeros(n_classes))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.drop = nn.Dropout(drop)

    def forward(self, feats, pos, mask=None, return_attention=False):
        # feats (B, K, F)  pos (B, K) slice indices  mask (B, K) bool
        h = self.norm(feats + self.pos(pos))
        a = self.att(h)                                         # (B, K, 12)
        if mask is not None:
            a = a.masked_fill(~mask.unsqueeze(-1), float("-inf"))
        a = torch.softmax(a.float(), dim=1).to(h.dtype)         # sums to 1 over windows
        pooled = torch.einsum("bkn,bkf->bnf", a, self.drop(h))  # (B, 12, F)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return (logits, a) if return_attention else logits


class KneeMILModel(nn.Module):
    def __init__(self, arch="convnext_tiny", n_classes=12, pretrained=True):
        super().__init__()
        self.backbone = build_backbone(arch, pretrained)
        self.head = PerFindingAttentionHead(self.backbone.num_features, n_classes)

    def forward(self, x, pos, mask=None, return_attention=False):
        # x (B, K, 3, H, W) float, already normalised
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).view(B, K, -1)
        return self.head(f, pos, mask, return_attention)


def to_input(windows_uint8, device):
    """(B, K, 3, H, W) uint8 -> ImageNet-normalised float on the GPU."""
    x = windows_uint8.to(device, non_blocking=True).float().div_(255.0)
    B, K = x.shape[:2]
    x = x.flatten(0, 1)
    x = (x - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device)
    return x.view(B, K, *x.shape[1:])


def gpu_augment(x):
    """Mild augmentations, identical for all windows of a study.
    No horizontal flip (it would swap medial and lateral).
    x : (B, K, 3, H, W) float."""
    B, K, C, H, W = x.shape
    dev = x.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * np.deg2rad(8)
    sc = 1 + (torch.rand(B, device=dev) * 2 - 1) * 0.10
    tx = (torch.rand(B, device=dev) * 2 - 1) * 0.05
    ty = (torch.rand(B, device=dev) * 2 - 1) * 0.05
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)  # (B, 2, 3)
    theta = theta.repeat_interleave(K, 0)
    flat = x.flatten(0, 1)
    grid = F.affine_grid(theta, flat.shape, align_corners=False)
    flat = F.grid_sample(flat, grid, align_corners=False, padding_mode="zeros")
    contrast = (1 + (torch.rand(B, device=dev) * 2 - 1) * 0.15).repeat_interleave(K).view(-1, 1, 1, 1)
    bright = ((torch.rand(B, device=dev) * 2 - 1) * 0.15).repeat_interleave(K).view(-1, 1, 1, 1)
    flat = flat * contrast + bright
    return flat.view(B, K, C, H, W)


@torch.no_grad()
def predict_volume(model, vol, mask, device, centers_fn, max_windows=64):
    """Predict 12 probabilities for a (64, H, W) uint8 volume using all valid windows."""
    centers = centers_fn(mask)
    if len(centers) == 0:
        return None
    centers = centers[:max_windows]
    win = np.stack([vol[c - 1:c + 2] for c in centers])[None]          # (1, K, 3, H, W)
    x = to_input(torch.from_numpy(win), device)
    pos = torch.from_numpy(centers)[None].to(device)
    try:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(x, pos)
    except RuntimeError:
        logits = model(x.float(), pos)
    return torch.sigmoid(logits.float())[0].cpu().numpy()

In [ ]:
import glob, os, sys, subprocess

def _walk_inputs(max_depth=5):
    """Walk /kaggle/input without ever entering the competition data (820,000 DICOM files)."""
    base = "/kaggle/input"
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        dirs[:] = [d for d in dirs if d != "competitions" and "rsna-knee" not in d
                   and not d.endswith("_series") and depth < max_depth]
        yield root, dirs, files

def find(name):
    """Path of a file produced by another notebook attached as input (any mount depth)."""
    for root, dirs, files in _walk_inputs():
        if name in files:
            return os.path.join(root, name)
    p = f"/kaggle/working/{name}"
    return p if os.path.exists(p) else None

print("Attached inputs:")
for root, dirs, files in _walk_inputs(max_depth=3):
    if files:
        print("  ", root, "->", sorted(files)[:6])

wheels = [os.path.join(r, f) for r, d, fs in _walk_inputs() if os.path.basename(r) == "wheels" for f in fs if f.endswith(".whl")]
print(len(wheels), "wheels")
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
common = find("knee_common.py")
if common is None:
    raise FileNotFoundError("knee_common.py not found: attach the K1 output (Add Input > Your Work > K1 series inventory).")
sys.path.insert(0, os.path.dirname(common))
print("shared code:", common)

sys.path.insert(0, "/kaggle/working")
import knee_common as kc, knee_model as km
import json, time, copy, math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

CFG = dict(arch="convnext_tiny", size=224, k_train=24, batch=4, accum=2, epochs=8,
           lr_backbone=3e-5, lr_head=1e-4, wd=0.02, warmup=0.05, ema=0.998, seed=7)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
if not torch.cuda.is_available():
    raise RuntimeError("No GPU in this session: set Accelerator to GPU T4 x2 or GPU P100 (Session options), then Save Version again.")
DEV = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
TARGETS = kc.TARGETS

COMP = kc.find_comp_dir()
train = pd.read_csv(f"{COMP}/train.csv")
V = np.load(find("volumes_224.npy"), mmap_mode="r")
M = np.load(find("masks.npy"))
idx = pd.read_csv(find("index.csv"))
soft = pd.read_csv(find("soft_labels.csv"))

row_of = dict(zip(idx["StudyInstanceUID"], idx["row"]))
ok_rows = set(idx.loc[idx["n_valid"] >= 3, "row"])
gold = train[train[TARGETS].notna().any(axis=1)].copy()
gold["row"] = gold["StudyInstanceUID"].map(row_of)
soft["row"] = soft["StudyInstanceUID"].map(row_of)
soft = soft[soft["row"].isin(ok_rows) & ~soft["StudyInstanceUID"].isin(gold["StudyInstanceUID"])].reset_index(drop=True)

perm = np.random.RandomState(0).permutation(len(soft))
n_val = int(0.05 * len(soft))
val_df, tr_df = soft.iloc[perm[:n_val]], soft.iloc[perm[n_val:]]
print("train", len(tr_df), " internal val", len(val_df), " gold", len(gold))
json.dump({t: float(soft[t].mean()) for t in TARGETS}, open("/kaggle/working/prior.json", "w"))
json.dump(CFG, open("/kaggle/working/config.json", "w"))

In [ ]:
class KneeDS(Dataset):
    def __init__(self, df, k):
        self.rows = df["row"].astype(int).values
        self.y = df[TARGETS].values.astype(np.float32)
        self.k = k
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        vol, mask = np.asarray(V[r]), M[r]
        c = kc.window_centers(mask)
        valid = np.ones(self.k, bool)
        if len(c) >= self.k:
            c = np.sort(np.random.choice(c, self.k, replace=False))
        else:
            pad = self.k - len(c); valid[len(c):] = False
            c = np.concatenate([c, np.full(pad, c[0])])
        win = np.stack([vol[j - 1:j + 2] for j in c])
        return torch.from_numpy(win), torch.from_numpy(c), torch.from_numpy(valid), torch.from_numpy(self.y[i])

dl = DataLoader(KneeDS(tr_df, CFG["k_train"]), batch_size=CFG["batch"], shuffle=True, num_workers=4,
                pin_memory=True, drop_last=True, persistent_workers=True)

model = km.KneeMILModel(CFG["arch"], pretrained=True).to(DEV)
ema = copy.deepcopy(model).eval()
for p in ema.parameters():
    p.requires_grad_(False)
opt = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CFG["lr_backbone"]},
    {"params": model.head.parameters(), "lr": CFG["lr_head"]},
], weight_decay=CFG["wd"])
steps = CFG["epochs"] * len(dl) // CFG["accum"]
warm = int(CFG["warmup"] * steps)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
scaler = torch.cuda.amp.GradScaler()
print(len(dl), "batches per epoch,", steps, "optimizer steps")

In [ ]:
def evaluate(net, df, binarize=False):
    net.eval(); P = []
    for r in df["row"].astype(int):
        p = km.predict_volume(net, np.asarray(V[r]), M[r], DEV, kc.window_centers)
        P.append(p if p is not None else np.full(12, np.nan))
    P = np.array(P); Y = df[TARGETS].fillna(0).values
    if binarize:
        Y = (Y >= 0.5).astype(int)
    aucs = [roc_auc_score(Y[:, j], P[:, j]) if 0 < Y[:, j].sum() < len(Y) and not np.isnan(P[:, j]).any() else np.nan
            for j in range(12)]
    return np.nanmean(aucs), aucs, P

def bootstrap_macro_auc(Y, P, n=1000, seed=0):
    rs, out = np.random.RandomState(seed), []
    for _ in range(n):
        i = rs.randint(0, len(Y), len(Y)); a = []
        for j in range(Y.shape[1]):
            if 0 < Y[i, j].sum() < len(i):
                a.append(roc_auc_score(Y[i, j], P[i, j]))
        if a:
            out.append(np.mean(a))
    return np.mean(out), np.percentile(out, [2.5, 97.5])

In [ ]:
step, t0 = 0, time.time()
ema_start = CFG["epochs"] // 3
history = []
for ep in range(CFG["epochs"]):
    model.train(); run = 0.0
    for b, (win, pos, valid, y) in enumerate(dl):
        x = km.gpu_augment(km.to_input(win, DEV))
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x, pos.to(DEV), valid.to(DEV))
        loss = F.binary_cross_entropy_with_logits(logits.float(), y.to(DEV)) / CFG["accum"]
        scaler.scale(loss).backward()
        run += loss.item() * CFG["accum"]
        if (b + 1) % CFG["accum"] == 0:
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            d = CFG["ema"] if ep >= ema_start else 0.0          # before ema_start the EMA simply copies the model
            with torch.no_grad():
                for pe, pm in zip(ema.state_dict().values(), model.state_dict().values()):
                    if pe.dtype.is_floating_point:
                        pe.mul_(d).add_(pm.detach(), alpha=1 - d)
                    else:
                        pe.copy_(pm)
        if (b + 1) % 200 == 0:
            print(f"  ep {ep+1} batch {b+1}/{len(dl)} loss {run/(b+1):.4f} {(time.time()-t0)/60:.0f} min")
    g_auc, g_aucs, _ = evaluate(ema, gold)
    v_auc, _, _ = evaluate(ema, val_df, binarize=True)
    history.append(dict(epoch=ep + 1, loss=run / len(dl), gold_auc=g_auc, val_auc=v_auc))
    print(f"EPOCH {ep+1}  loss {run/len(dl):.4f}  gold AUC {g_auc:.4f}  internal val AUC {v_auc:.4f}  {(time.time()-t0)/60:.0f} min")
    torch.save(ema.state_dict(), "/kaggle/working/model_final.pt")   # checkpoint after every epoch
pd.DataFrame(history).to_csv("/kaggle/working/history.csv", index=False)

In [ ]:
# Final report on the 58 gold studies: per-class AUC and bootstrap interval
g_auc, g_aucs, P = evaluate(ema, gold)
Y = gold[TARGETS].fillna(0).values
print(pd.DataFrame({"finding": TARGETS, "positives": Y.sum(0).astype(int), "AUC": np.round(g_aucs, 3)}).to_string(index=False))
m, ci = bootstrap_macro_auc(Y, P)
print(f"Gold macro AUC {g_auc:.4f}  bootstrap {m:.4f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
pd.DataFrame(P, columns=TARGETS).assign(StudyInstanceUID=gold["StudyInstanceUID"].values).to_csv(
    "/kaggle/working/gold_preds.csv", index=False)

In [ ]:
# Ensemble check on the 58 gold studies: K4 (seed 42) + K4b (seed 7), mean of probabilities
prev = None
for root, dirs, files in _walk_inputs():
    if "gold_preds.csv" in files and "k4-training" in root:
        prev = os.path.join(root, "gold_preds.csv")
if prev is None:
    print("K4 output not attached: ensemble check skipped.")
else:
    G1 = pd.read_csv(prev).set_index("StudyInstanceUID").loc[gold["StudyInstanceUID"], TARGETS].values
    PE = (G1 + P) / 2
    a1 = [roc_auc_score(Y[:, j], G1[:, j]) for j in range(12)]
    ae = [roc_auc_score(Y[:, j], PE[:, j]) for j in range(12)]
    print(pd.DataFrame({"finding": TARGETS, "K4": np.round(a1, 3), "K4b": np.round(g_aucs, 3), "ensemble": np.round(ae, 3)}).to_string(index=False))
    print(f"macro AUC  K4 {np.mean(a1):.4f}   K4b {np.mean(g_aucs):.4f}   ensemble {np.mean(ae):.4f}")
    m, ci = bootstrap_macro_auc(Y, PE)
    print(f"ensemble bootstrap {m:.4f}  95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")